# KATHE 2026 — Kashmiri Machine Translation Pipeline

This notebook runs the complete **KATHE 2026** English → Kashmiri translation pipeline on GPU (Google Colab / Kaggle GPU).

### Pipeline Steps:
1. **Environment Setup** (Dependencies & Hugging Face authentication)
2. **Script Verification** (`sample_submission.csv` target script check)
3. **Smoke Test** (20 rows on GPU)
4. **Full Batch Translation** (Generates `submission.csv` using IndicTrans2 1B)
5. **(Optional) LoRA Fine-tuning** (Stretch goal on BPCC en-kas)

## 1. Setup & Environment

In [ ]:
# Install PyTorch CUDA and dependencies
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

import os
from getpass import getpass

# Prompt for Hugging Face Token (accept model terms at https://huggingface.co/ai4bharat/indictrans2-en-indic-1B first)
if "HF_TOKEN" not in os.environ:
    hf_token = getpass("Enter your Hugging Face Access Token: ")
    os.environ["HF_TOKEN"] = hf_token

## 2. Check Target Script (`kas_Arab` vs `kas_Deva`)
Upload `sample_submission.csv` from Kaggle Data tab to verify the expected target script.

In [ ]:
!python check_script.py sample_submission.csv

## 3. GPU Smoke Test (20 rows)

In [ ]:
!python inference.py --limit 20 --output /tmp/smoke.csv --fp16
!head -n 5 /tmp/smoke.csv

## 4. Full Translation Run (generates `submission.csv`)

In [ ]:
!python inference.py \
  --input data/englishdev.csv \
  --output submission.csv \
  --model ai4bharat/indictrans2-en-indic-1B \
  --tgt-lang kas_Arab \
  --batch-size 16 \
  --fp16

## 5. (Optional) LoRA Fine-Tuning

In [ ]:
# Step A: Train LoRA adapter
!python finetune.py --max-samples 50000 --epochs 1 --output-dir out/lora-kas

# Step B: Run inference using LoRA weights
!python inference.py --lora out/lora-kas --output submission_lora.csv --fp16